<a href="https://colab.research.google.com/github/chight1/PaddlePaddle-Practice/blob/main/HousingPrices.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install paddlepaddle

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

In [ ]:
features_to_drop = ['Alley', 'FireplaceQu', 'PoolQC', 'MiscFeature', 'Fence', 'MasVnrType', 'LotFrontage', 'GarageQual', 'GarageCond', 'BsmtCond']
train_df = train_df.drop(features_to_drop, axis=1)
test_df = test_df.drop(features_to_drop, axis=1)

cat_features = ['BsmtQual', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'GarageType', 'GarageFinish']
train_df[cat_features] = train_df[cat_features].fillna('NA')
test_df[cat_features] = test_df[cat_features].fillna('NA')

median_masvnr = train_df['MasVnrArea'].median()
median_garageyr = train_df['GarageYrBlt'].median()
train_df['MasVnrArea'] = train_df['MasVnrArea'].fillna(median_masvnr)
train_df['GarageYrBlt'] = train_df['GarageYrBlt'].fillna(median_garageyr)
test_df['MasVnrArea'] = test_df['MasVnrArea'].fillna(median_masvnr)
test_df['GarageYrBlt'] = test_df['GarageYrBlt'].fillna(median_garageyr)

mode_electrical = train_df['Electrical'].mode()[0]
train_df['Electrical'] = train_df['Electrical'].fillna(mode_electrical) ## only one null
test_df['Electrical'] = test_df['Electrical'].fillna(mode_electrical)

for col in ['MSZoning', 'Utilities', 'Exterior1st', 'Exterior2nd', 'KitchenQual', 'Functional', 'SaleType']:
    test_df[col] = test_df[col].fillna(train_df[col].mode()[0])

for col in ['BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
            'BsmtFullBath', 'BsmtHalfBath', 'GarageCars', 'GarageArea']:
    test_df[col] = test_df[col].fillna(train_df[col].median())

# get null counts of each column
null_counts = test_df.isnull().sum()
null_counts[null_counts > 0]

In [ ]:
numeric_cols = train_df.select_dtypes(include=['float64', 'int64']).columns
scaler = StandardScaler()
train_df[numeric_cols] = scaler.fit_transform(train_df[numeric_cols])

In [ ]:
print("Features before:", train_df.shape[1])
train_df = pd.get_dummies(train_df)
print("Features after:", train_df.shape[1])

In [ ]:
# correlation matrix
corr_matrix = train_df.corr().abs()

# Keep features with correlation >= 0.3 to SalePrice
strong_corr = corr_matrix['SalePrice'][corr_matrix['SalePrice'] >= 0.4].index

# Drop all weak corr features
train_df_reduced = train_df[strong_corr]

print("Original features:", train_df.shape[1])
print("Reduced features:", train_df_reduced.shape[1])